# Runtime

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model
from rich import print as rprint

In [3]:
model_free = init_chat_model("openai/gpt-oss-20b",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=10000, temperature=0.0)

model_basic = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

model_medium = init_chat_model("openai/gpt-5.6-luna",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_advanced = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_safety = init_chat_model("nvidia/nemotron-3.5-content-safety:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)



In [4]:

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent, AgentState
from langchain_core.tools import tool
from langchain.tools import tool as tool_rt, ToolRuntime
from dataclasses import dataclass


In [5]:
@dataclass
class CineBotContext:
  user_name:str
  #geo_state:str # Delhi or Bangalore or Chennai




In [6]:
agent_with_context = create_agent(
    model=model_advanced,
    tools=[],
    context_schema=CineBotContext,   # declares the SHAPE of context this agent expects
)

In [7]:
result = agent_with_context.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    context=CineBotContext(user_name="Priya"),   # injected at invocation time
)

In [8]:
rprint(result)

{
    'messages': [
        HumanMessage(
            content="What's my name?",
            additional_kwargs={},
            response_metadata={},
            id='1537511d-663c-47a1-9e03-82e4134ebf9d'
        ),
        AIMessage(
            content='I don’t know your name—you haven’t told me.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1787444943-SXdeV8oOyCZMjW8VGf5s',
                'created': 1787444943,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0004234,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.000114,
                    'upstream_inference_prompt_cost': 0.0003094,
                    'upstream_inference_cost': 0.0004234
                }
            },
            id='lc_run--01a02c05-5596-7453-9035-d446ffd6200a-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1547,
                'output_tokens': 95,
                'total_tokens': 1642,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 30}
            }
        )
    ]
}

# Runtime Inside Tools : ToolRuntime

In [9]:
@dataclass
class CustomerContext:
  user_id:str


In [10]:
from langgraph.store.memory import InMemoryStore

loyalty_store=InMemoryStore()

In [11]:
@tool
def fetch_customer_preferences(runtime: ToolRuntime[CustomerContext])-> str:
  "Feth the customer's Saved Preferences from a long term Memory"
  user_id=runtime.context.user_id
  preferences = "No Preferences"

  if runtime.store:
    if memory := runtime.store.get(('users'),user_id):
      preferences = memory.value['preferences']

  return preferences

In [12]:
pref_agent = create_agent(
    model=model_advanced,
    tools=[fetch_customer_preferences],
    context_schema=CustomerContext,
    store=loyalty_store,
)

# MiddleWare

In [13]:
#NodeStyle ( Before_model, after_model)
# Wrap_style_hooks ( wrap_model_call)

# Node Style Hooks

In [14]:
from langchain.agents.middleware import before_model, after_model


In [15]:
@before_model
def log_before_model(a,b):
  pass

In [16]:
logged_agent = create_agent(model=model_advanced, tools=[fetch_customer_preferences], middleware=[log_before_model])


In [ ]:
result = logged_agent.invoke({"messages": [("user", "Get user preferences")]})

# Wrap Style Hooks

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable


In [ ]:
@wrap_model_call
def dynamic_greeting_prompt(request: ModelRequest,handler: Callable[[ModelRequest], ModelResponse])->ModelResponse:
  response = handler(request)
  return response

In [ ]:

wrap_style_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[],
    middleware=[dynamic_greeting_prompt],
    context_schema=CineBotContext,
)

In [ ]:
result = wrap_style_agent.invoke(
    {"messages": [("user", "What should I call you, and who am I?")]},
    context=CineBotContext(user_name="Rohan"),
)

# Dynamic Prompting

In [ ]:
from langchain.agents.middleware import dynamic_prompt


In [ ]:
@dataclass
class ClaudeContext:
  user_specific_instructions: str

In [ ]:
@dynamic_prompt
def personalize_the_prompt(request:ModelRequest):
  user_name = request.runtime.context.user_name
  return f"{Claude_Instructions} + {request.runtime.context.user_specific_instructions}"

In [ ]:
@before_model
def auth_gate(state:AgentState,runtime:Runtime)-> dict|None:
  "Block Unauthenticated users"
  server = runtime.server_info
  if server is not None:
    raise ValueError("Unauthenticated User")

  print(f"[AUTH] Passed the check for user {runtime.context.user_name} and threadID {runtime.execution_info.thread_id}")


NameError: name 'Runtime' is not defined

# Human in the Loop ( In Depth )

In [19]:

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
# model = init_chat_model("openai:gpt-5-mini")

In [20]:
from rich import print

In [21]:
@tool
def cancel_booking(booking_id: str,runtime:ToolRuntime) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."

In [22]:
unguarded_agent = create_agent(model=model_advanced, tools=[cancel_booking])
result = unguarded_agent.invoke({"messages": [("user", "Cancel my booking BK1042")]})

In [23]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Cancel my booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='61a79447-1944-44c0-b9c4-1d390d1950f5'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqij-P9ljYkKfBL3PhCk1wbsHqRV-ikUDvlqLAU1W7BfvQuwpatIc_4WDt5viMJtyB1DKGKZupAnzAcqqfdgOgITukPA7eBuz31ebQAiH2kV
w-D6owuqKP-L2oPKshmGODaRbvRDLcAVk6j-bXAdqmK5m0abIe7eFfD519B8P1UZ5XtslO7lFmnVwK1-oDD7ZDrzezRgxKvghF17Cd8MHMahhZP5_J4
JTrY0aL3VED_KAXuSDkA6jI_SBWFxM3N9meLlr74LM3OqSB6Y1z_JYiJGhoND5Fl78hB9fUSpzZtdbkSJXDfkKnUdd7CW6VvHCtzbnuexyeqK0JPgET
7In7KOLrlgro1n0bbZRdbsAwWxSwXNW-dibzW6kNhFPP6Al3RV6j6Ub4k15r2PgOc8K5D1sbIOG11KhhiXMbWrRa4JP8LSGqt0AwPl-8eheXN5iMf7g
PveiohsPJfiFEVQaqYL_j6AWRC10YnS6NhnhH28oCRXC1I3XalGZ5x4bzM3oqHlEKfwvxyPCtyIpnfGqVVMArOZWmA-dsTjSJvrAFui2CVvrjZYen0B
djH94eVDjrfHopAFpG_E_KfMJ9xOleJUWUJtYUy-x8T3s20Cci0Nyu6t66gM51-hTp3Z1eFZ8rfW640DO3Ge-z2TdljU_et653f1D-lTFCs2U_7msV9
Iprs0UwOJ2BU566AdG-N3eG7ZGi4OY4JibqPz6CWpZAZGIqPTQWuuuzqZ3xIYikqsFKrXAFihjvQkyGsbWQ0uF1YN4hBYmhaiet1hB5AkVmoM0BqpME
b5IiOf7Vwv7LepJ_5IsAB3SchS-PEI7rNphbTYvwlPry7Z7PvenEatBBoPdOwpv_qcwluIk_m50dgi0sr5h2Hrbi6B8G9ywkqQUWoDCacE56QmIEjY8
zZ0eR4th17gyqJ34VsC_0emEMThacSNc8Kq1mrGIdvUPYff0WV_OaiHtplOfrsMTDq7pA-XCHKNMCA4EVD5eNT3KN5VqQEl1D1J_xOECq_1dFYXoDY4
VEMsz76anMqbWtZfYNRit-lTZ5MHUfnXeLVOTkS_pxerw3tfCsUglzfo1PFX9rDPRRBr54ZI_FKrzHfUJrhDIx94tr8qppf8Q-RdAUXcUhCkhfZvFUz
P71MjqITqJ6bFr5ZSsSuJA-5RhSkXi-pZGR4cTN2uO4_gnBDsc4y7Conuu6oMgnKfBQYcysu3MIaLyQN3tzKeGrA1uTtm11YIzyZC2l_IpPJc7tPeFS
rwzkow4Wb0spf8j2BlCLcGF8K35bALuLFvpFxDkXa_NxDVA37yHWx3PeTRv_OJ2r9ucvRuvmrY1xjZZ8HXXewXpMJlbWzq1frW-pBsE_kHOpsk_mG1J
3xjeG7W6knu2T-FaL-VCU8mFWGcRVx43utO7GxvjwJScP8-rCuIoFsYAXk7oeiA==.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuY
S1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_012f2726caf093e5016a8a3f8f0c1887d18b5bc4e5d19614b0',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1787445132-4xbDJxlQ5ah86b6FtqrQ',
                'created': 1787445132,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0005242,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0001848,
                    'upstream_inference_prompt_cost': 0.0003394,
                    'upstream_inference_cost': 0.0005242
                }
            },
            id='lc_run--01a02c08-395b-7611-937f-3726282bbad2-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'id': 'call_CEMRUnSuD7To6oKa7AwTEnU6',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1697,
                'output_tokens': 154,
                'total_tokens': 1851,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 68}
            }
        ),
        ToolMessage(
            content='Booking BK1042 cancelled.',
            name='cancel_booking',
            id='1b90d5a8-9654-4085-bc02-1f9a446bdd71',
            tool_call_id='call_CEMRUnSuD7To6oKa7AwTEnU6'
        ),
        AIMessage(
            content='Booking **BK1042** has been cancelled.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
         

In [ ]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Cancel my booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='55662253-1f48-490a-885a-2420f840f369'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 154,
                    'prompt_tokens': 134,
                    'total_tokens': 288,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 128,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EFXwcD74f67ALpmrV3loAyJKrTgRz',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a027cc-84c5-7853-8e13-48799c06d04e-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'id': 'call_nPYwKSLt1EGN4NnLhqUp213w',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 134,
                'output_tokens': 154,
                'total_tokens': 288,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 128}
            }
        ),
        ToolMessage(
            content='Booking BK1042 cancelled.',
            name='cancel_booking',
            id='5f79f1bc-b0a7-49c8-b7bf-cdc93824165e',
            tool_call_id='call_nPYwKSLt1EGN4NnLhqUp213w'
        ),
        AIMessage(
            content='Done — your booking BK1042 has been cancelled.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 14,
                    'prompt_tokens': 171,
                    'total_tokens': 185,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EFXwelyIkGCz1DMAKdF3uun8KcW2A',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a027cc-8ed1-72c0-b4bd-f49dfce6299f-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 171,
                'output_tokens': 14,
                'total_tokens': 185,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}

In [ ]:
result = unguarded_agent.invoke({"messages": [("user", "Hi I am Mayank")]})

In [ ]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Hi I am Mayank',
            additional_kwargs={},
            response_metadata={},
            id='2d8c4ec4-dbfb-460c-9cff-2d10ed0d2302'
        ),
        AIMessage(
            content='Hi Mayank — nice to meet you. How can I help you today? (I can answer questions, draft or edit
text, help with planning, check or cancel bookings, run calculations, or anything else you need.)',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 118,
                    'prompt_tokens': 133,
                    'total_tokens': 251,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EFXyVmAKfEkaUBgG7h1cKWJXl1kuM',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a027ce-4dd7-7592-898c-ffde6540f6d3-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 133,
                'output_tokens': 118,
                'total_tokens': 251,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        )
    ]
}

In [ ]:
result = unguarded_agent.invoke({"messages": [("user", "Who am I?")]})

In [ ]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Who am I?',
            additional_kwargs={},
            response_metadata={},
            id='3f827588-023f-4145-90cf-0ab3751ade5e'
        ),
        AIMessage(
            content='I can’t read your mind or see your identity from here — but I can help you discover or 
describe who you are. Do you mean it philosophically (your purpose), practically (your roles, job, personality), or
playfully (a riddle)?\n\nChoose one, or pick from these options and I’ll proceed:\n\n- Give me some facts (age, 
job, interests, goals) and I’ll write a clear “Who you are” summary.\n- I’ll ask a short set of reflective 
questions and synthesize your answers into a profile.\n- I can run a quick personality-style quiz (Big Five / brief
Enneagram-style).\n- I can give exercises to explore values, strengths, and purpose.\n- I can answer creatively 
(poem, short story, or imaginative “you are”).\n\nIf you want to start right now, answer any or all of these four 
quick prompts and I’ll make a short profile from them:\n\n1) Three roles you currently fill (e.g., parent, student,
designer).  \n2) Three strengths or things others compliment you about.  \n3) Three values that matter most to you 
(e.g., honesty, freedom, learning).  \n4) One thing that excites you and one thing that drains you.\n\nReply 
however you like and I’ll take it from there.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 596,
                    'prompt_tokens': 132,
                    'total_tokens': 728,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 320,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EFY343Nb4t82ydG2fN5u44s5HTxde',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a027d2-9f57-7681-bc4e-7cac3d97a346-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 132,
                'output_tokens': 596,
                'total_tokens': 728,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 320}
            }
        )
    ]
}

In [ ]:
result.interrupts

AttributeError: 'dict' object has no attribute 'interrupts'

In [24]:
@tool
def send_booking_confirmation(booking_id: str, email: str) -> str:
    """Send a booking confirmation email. Safe, no approval needed."""
    return f"Confirmation for {booking_id} sent to {email}."


In [25]:
guarded_agent = create_agent(
    model=model_advanced,
    tools=[cancel_booking, send_booking_confirmation],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "cancel_booking": True,               # all four decisions allowed (default config)
                "send_booking_confirmation": False,    # safe operation, auto-approved, never pauses
            },
            description_prefix="CineBot action pending your approval",
        ),
    ],
    checkpointer=InMemorySaver(),   # REQUIRED -- HITL needs to pause and later resume
)

In [26]:
config = {"configurable": {"thread_id": "hitl-demo-111"}}

In [27]:
result = guarded_agent.invoke(
    {"messages": [("user", "Cancel booking BK1042")]},
    config=config,
    version="v2",   # the current, recommended invoke pattern for reading interrupts
)

In [28]:
print(result)

GraphOutput(
    value={
        'messages': [
            HumanMessage(
                content='Cancel booking BK1042',
                additional_kwargs={},
                response_metadata={},
                id='7c0e4a8c-46e5-4e5f-95af-605d68326e56'
            ),
            AIMessage(
                content='',
                additional_kwargs={
                    'reasoning_details': [
                        {
                            'data': 
'gAAAAABqij_uSroA_vTEVgsBjw82wIzi9WVcsJNA5WOpPURHm6knvs5I3mOO-9j7z3nnidO3PJ5ONVRJc6RzFkrGq7vgNtQypLx975ZnhjGwIyHX0f
A-1lsM2KJwXPm8ZdPeiBzBKcHRVuS6WwBoaQtSSmLAGwBzZ2W8josH2OxgsEp8s8l3m0CFcIgOnXkmxJA8KCE4BHV57GHVPO0z5uFcdb6uGvyi-zYX6
J4Yiw9xzIRdie58nSB9MLGKEacJXc3BJs78kT913TRVQ4GGLWe-7vfbGWleT7AeOGwuheUMv2kx_XvYB18lUaUok_iDlAqt8o3w_uERTloAcn1F4Dfd
4cgZk0mGva8ORHcgypEJ8Tkd8HtlEGV9BLjbex1tAVSVr1L_W9Id43mEGnDiCNFUTefM8Xj-YpIuBcwvwBQTEeGMdTYzdFTge9Rp0iR9qKR5sBh8C0G
tpSgMc4V1lcaQj5Ia1f95zf4gm1ZZFEkjg2XFRxOmBUgkB-rO7iLAwlDSUQFDwOc_1kqGDstJr5boWfiotCSRK7xF56SI4phEIPpZbOdtTG9-Px092s
f3nHUZ_AHQ8auVU7hzmrlkHLreUTczkztF9pkavqpmxKv1eCUZdLBQMogB71pqPmNN08xC8vdvJanEfdQZzmcCev70v5O4ycI3cd-TFpGxKqPyc2vG4
Rb0FSIbZCT52_tV1lPGAfS5VsBWSYLWBtSeVzllk_9WFIYDMYRu7so4bdeRkoDeuGNgCgOnbqvqCw_Ft8O1xGJkfhWxsY5knfxqOSFiWU0FWWIJLZB9
IRDZYbgNbSSwsKGB10_nwlamsTM37IntIp1QtNTv6RBs3gSNKFF68yktDd5DPThewNOg-KGXNeM_mlkmHDM1CeT6a5PeWd1gUD1aDz6CICcdRn1s-eS
klnIMIjl9aa7E00N-ISoJQ5QDtaOTEX6pTpR7UWw3DKzFD4SGhQ73EADTy6ygL68r9kMTLm50qNDEcCFQ7oxHJoU6JR7sQ2Ky1TguMbdKwW49FFX03d
civpXJQGZIQkBjuLBxHT-9VHLz6bS8HyJs_0b0EKBFRwE2kzTh3h6vtoyGSqKGNJVjzLkydO2V7OvvA5YK7fc7VAg0UiEhv-ugsgaXsYDwQNVVVxi9T
nrcNPTicF-YrltXK2zZBnZZxQcGWa8bX03YQ3U9X-FlB8QxphF_9bYBV_-hPwXtSf52kg5ssp7jF0QmBw5sf1SiP70291QUTijBIcKPkn45331pMNRZ
5bmKzvVrCN3Q77Nv4tqXr0apVXk7PXSj7VoVPovhfBljNm5rHiCSXTIXmVSK74Zlq7gI2RvrY5skXz0iPB8K7LE2mz1b85kj3Y_3iwr3VDBQOq44rxT
u9_63Y6naF0kvB2qjDH2VMpmsFJ48fXuSUQbOk3bUKVdpJ-q1YCDSqxj8T-xn5w==.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuY
S1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                            'type': 'reasoning.encrypted',
                            'format': 'openai-responses-v1',
                            'id': 'rs_00e4107ce4414b97016a8a3fedff8887d19a264d2dadb01c37',
                            'index': 0
                        }
                    ]
                },
                response_metadata={
                    'model_name': 'openai/gpt-5.6-luna-pro',
                    'id': 'gen-1787445226-ohuNlqAxLMRbmFKbrlaL',
                    'created': 1787445226,
                    'object': 'chat.completion',
                    'finish_reason': 'tool_calls',
                    'logprobs': None,
                    'model_provider': 'openrouter',
                    'cost': 0.0005836,
                    'cost_details': {
                        'upstream_inference_completions_cost': 0.0002256,
                        'upstream_inference_prompt_cost': 0.000358,
                        'upstream_inference_cost': 0.0005836
                    }
                },
                id='lc_run--01a02c09-aa2b-7a83-ad50-98e83cdf3896-0',
                tool_calls=[
                    {
                        'name': 'cancel_booking',
                        'args': {'booking_id': 'BK1042'},
                        'id': 'call_zrkezAN7jMY1RBB71WBcLbGZ',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 1790,
                    'output_tokens': 188,
                    'total_tokens': 1978,
                    'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                    'output_token_details': {'reasoning': 102}
                }
            )
        ]
    },
    interrupts=(
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'cancel_booking',
                        'arg

In [29]:
result.interrupts

(Interrupt(value={'action_requests': [{'name': 'cancel_booking', 'args': {'booking_id': 'BK1042'}, 'description': "CineBot action pending your approval\n\nTool: cancel_booking\nArgs: {'booking_id': 'BK1042'}"}], 'review_configs': [{'action_name': 'cancel_booking', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='c454c903841b5bdfa6a694caed13174c'),)

# Command

In [30]:
resumed = guarded_agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ),
    config=config, # Same thread ID to resume the paused conversation
    version="v2",
)

In [31]:
print(resumed)

GraphOutput(
    value={
        'messages': [
            HumanMessage(
                content='Cancel booking BK1042',
                additional_kwargs={},
                response_metadata={},
                id='7c0e4a8c-46e5-4e5f-95af-605d68326e56'
            ),
            AIMessage(
                content='',
                additional_kwargs={
                    'reasoning_details': [
                        {
                            'data': 
'gAAAAABqij_uSroA_vTEVgsBjw82wIzi9WVcsJNA5WOpPURHm6knvs5I3mOO-9j7z3nnidO3PJ5ONVRJc6RzFkrGq7vgNtQypLx975ZnhjGwIyHX0f
A-1lsM2KJwXPm8ZdPeiBzBKcHRVuS6WwBoaQtSSmLAGwBzZ2W8josH2OxgsEp8s8l3m0CFcIgOnXkmxJA8KCE4BHV57GHVPO0z5uFcdb6uGvyi-zYX6
J4Yiw9xzIRdie58nSB9MLGKEacJXc3BJs78kT913TRVQ4GGLWe-7vfbGWleT7AeOGwuheUMv2kx_XvYB18lUaUok_iDlAqt8o3w_uERTloAcn1F4Dfd
4cgZk0mGva8ORHcgypEJ8Tkd8HtlEGV9BLjbex1tAVSVr1L_W9Id43mEGnDiCNFUTefM8Xj-YpIuBcwvwBQTEeGMdTYzdFTge9Rp0iR9qKR5sBh8C0G
tpSgMc4V1lcaQj5Ia1f95zf4gm1ZZFEkjg2XFRxOmBUgkB-rO7iLAwlDSUQFDwOc_1kqGDstJr5boWfiotCSRK7xF56SI4phEIPpZbOdtTG9-Px092s
f3nHUZ_AHQ8auVU7hzmrlkHLreUTczkztF9pkavqpmxKv1eCUZdLBQMogB71pqPmNN08xC8vdvJanEfdQZzmcCev70v5O4ycI3cd-TFpGxKqPyc2vG4
Rb0FSIbZCT52_tV1lPGAfS5VsBWSYLWBtSeVzllk_9WFIYDMYRu7so4bdeRkoDeuGNgCgOnbqvqCw_Ft8O1xGJkfhWxsY5knfxqOSFiWU0FWWIJLZB9
IRDZYbgNbSSwsKGB10_nwlamsTM37IntIp1QtNTv6RBs3gSNKFF68yktDd5DPThewNOg-KGXNeM_mlkmHDM1CeT6a5PeWd1gUD1aDz6CICcdRn1s-eS
klnIMIjl9aa7E00N-ISoJQ5QDtaOTEX6pTpR7UWw3DKzFD4SGhQ73EADTy6ygL68r9kMTLm50qNDEcCFQ7oxHJoU6JR7sQ2Ky1TguMbdKwW49FFX03d
civpXJQGZIQkBjuLBxHT-9VHLz6bS8HyJs_0b0EKBFRwE2kzTh3h6vtoyGSqKGNJVjzLkydO2V7OvvA5YK7fc7VAg0UiEhv-ugsgaXsYDwQNVVVxi9T
nrcNPTicF-YrltXK2zZBnZZxQcGWa8bX03YQ3U9X-FlB8QxphF_9bYBV_-hPwXtSf52kg5ssp7jF0QmBw5sf1SiP70291QUTijBIcKPkn45331pMNRZ
5bmKzvVrCN3Q77Nv4tqXr0apVXk7PXSj7VoVPovhfBljNm5rHiCSXTIXmVSK74Zlq7gI2RvrY5skXz0iPB8K7LE2mz1b85kj3Y_3iwr3VDBQOq44rxT
u9_63Y6naF0kvB2qjDH2VMpmsFJ48fXuSUQbOk3bUKVdpJ-q1YCDSqxj8T-xn5w==.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuY
S1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                            'type': 'reasoning.encrypted',
                            'format': 'openai-responses-v1',
                            'id': 'rs_00e4107ce4414b97016a8a3fedff8887d19a264d2dadb01c37',
                            'index': 0
                        }
                    ]
                },
                response_metadata={
                    'model_name': 'openai/gpt-5.6-luna-pro',
                    'id': 'gen-1787445226-ohuNlqAxLMRbmFKbrlaL',
                    'created': 1787445226,
                    'object': 'chat.completion',
                    'finish_reason': 'tool_calls',
                    'logprobs': None,
                    'model_provider': 'openrouter',
                    'cost': 0.0005836,
                    'cost_details': {
                        'upstream_inference_completions_cost': 0.0002256,
                        'upstream_inference_prompt_cost': 0.000358,
                        'upstream_inference_cost': 0.0005836
                    }
                },
                id='lc_run--01a02c09-aa2b-7a83-ad50-98e83cdf3896-0',
                tool_calls=[
                    {
                        'name': 'cancel_booking',
                        'args': {'booking_id': 'BK1042'},
                        'id': 'call_zrkezAN7jMY1RBB71WBcLbGZ',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 1790,
                    'output_tokens': 188,
                    'total_tokens': 1978,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                    'output_token_details': {'reasoning': 102}
                }
            ),
            ToolMessage(
                content='Booking BK1042 cancelled.',
                name='cancel_booking',
                id='553d4861-7351-4f2d-8ee2-a804ef4ec8e7',
                tool_call_id='cal

# Edit

In [33]:
resumed_edit = guarded_agent.invoke(
    Command(resume={"decisions": [{
        "type": "edit",
        "edited_action": {"name": "cancel_booking", "args": {"booking_id": "BK1042"}},
    }]}),
    config=config, version="v2",
)

In [36]:
print(resumed_edit)

GraphOutput(
    value={
        'messages': [
            HumanMessage(
                content='Cancel booking BK1042',
                additional_kwargs={},
                response_metadata={},
                id='7c0e4a8c-46e5-4e5f-95af-605d68326e56'
            ),
            AIMessage(
                content='',
                additional_kwargs={
                    'reasoning_details': [
                        {
                            'data': 
'gAAAAABqij_uSroA_vTEVgsBjw82wIzi9WVcsJNA5WOpPURHm6knvs5I3mOO-9j7z3nnidO3PJ5ONVRJc6RzFkrGq7vgNtQypLx975ZnhjGwIyHX0f
A-1lsM2KJwXPm8ZdPeiBzBKcHRVuS6WwBoaQtSSmLAGwBzZ2W8josH2OxgsEp8s8l3m0CFcIgOnXkmxJA8KCE4BHV57GHVPO0z5uFcdb6uGvyi-zYX6
J4Yiw9xzIRdie58nSB9MLGKEacJXc3BJs78kT913TRVQ4GGLWe-7vfbGWleT7AeOGwuheUMv2kx_XvYB18lUaUok_iDlAqt8o3w_uERTloAcn1F4Dfd
4cgZk0mGva8ORHcgypEJ8Tkd8HtlEGV9BLjbex1tAVSVr1L_W9Id43mEGnDiCNFUTefM8Xj-YpIuBcwvwBQTEeGMdTYzdFTge9Rp0iR9qKR5sBh8C0G
tpSgMc4V1lcaQj5Ia1f95zf4gm1ZZFEkjg2XFRxOmBUgkB-rO7iLAwlDSUQFDwOc_1kqGDstJr5boWfiotCSRK7xF56SI4phEIPpZbOdtTG9-Px092s
f3nHUZ_AHQ8auVU7hzmrlkHLreUTczkztF9pkavqpmxKv1eCUZdLBQMogB71pqPmNN08xC8vdvJanEfdQZzmcCev70v5O4ycI3cd-TFpGxKqPyc2vG4
Rb0FSIbZCT52_tV1lPGAfS5VsBWSYLWBtSeVzllk_9WFIYDMYRu7so4bdeRkoDeuGNgCgOnbqvqCw_Ft8O1xGJkfhWxsY5knfxqOSFiWU0FWWIJLZB9
IRDZYbgNbSSwsKGB10_nwlamsTM37IntIp1QtNTv6RBs3gSNKFF68yktDd5DPThewNOg-KGXNeM_mlkmHDM1CeT6a5PeWd1gUD1aDz6CICcdRn1s-eS
klnIMIjl9aa7E00N-ISoJQ5QDtaOTEX6pTpR7UWw3DKzFD4SGhQ73EADTy6ygL68r9kMTLm50qNDEcCFQ7oxHJoU6JR7sQ2Ky1TguMbdKwW49FFX03d
civpXJQGZIQkBjuLBxHT-9VHLz6bS8HyJs_0b0EKBFRwE2kzTh3h6vtoyGSqKGNJVjzLkydO2V7OvvA5YK7fc7VAg0UiEhv-ugsgaXsYDwQNVVVxi9T
nrcNPTicF-YrltXK2zZBnZZxQcGWa8bX03YQ3U9X-FlB8QxphF_9bYBV_-hPwXtSf52kg5ssp7jF0QmBw5sf1SiP70291QUTijBIcKPkn45331pMNRZ
5bmKzvVrCN3Q77Nv4tqXr0apVXk7PXSj7VoVPovhfBljNm5rHiCSXTIXmVSK74Zlq7gI2RvrY5skXz0iPB8K7LE2mz1b85kj3Y_3iwr3VDBQOq44rxT
u9_63Y6naF0kvB2qjDH2VMpmsFJ48fXuSUQbOk3bUKVdpJ-q1YCDSqxj8T-xn5w==.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuY
S1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                            'type': 'reasoning.encrypted',
                            'format': 'openai-responses-v1',
                            'id': 'rs_00e4107ce4414b97016a8a3fedff8887d19a264d2dadb01c37',
                            'index': 0
                        }
                    ]
                },
                response_metadata={
                    'model_name': 'openai/gpt-5.6-luna-pro',
                    'id': 'gen-1787445226-ohuNlqAxLMRbmFKbrlaL',
                    'created': 1787445226,
                    'object': 'chat.completion',
                    'finish_reason': 'tool_calls',
                    'logprobs': None,
                    'model_provider': 'openrouter',
                    'cost': 0.0005836,
                    'cost_details': {
                        'upstream_inference_completions_cost': 0.0002256,
                        'upstream_inference_prompt_cost': 0.000358,
                        'upstream_inference_cost': 0.0005836
                    }
                },
                id='lc_run--01a02c09-aa2b-7a83-ad50-98e83cdf3896-0',
                tool_calls=[
                    {
                        'name': 'cancel_booking',
                        'args': {'booking_id': 'BK1042'},
                        'id': 'call_zrkezAN7jMY1RBB71WBcLbGZ',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 1790,
                    'output_tokens': 188,
                    'total_tokens': 1978,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                    'output_token_details': {'reasoning': 102}
                }
            ),
            ToolMessage(
                content='Booking BK1042 cancelled.',
                name='cancel_booking',
                id='553d4861-7351-4f2d-8ee2-a804ef4ec8e7',
                tool_call_id='cal

In [37]:
config3 = {"configurable": {"thread_id": "hitl-demo-3"}}
guarded_agent.invoke({"messages": [("user", "Cancel booking BK1042")]}, config=config3, version="v2")

resumed_reject = guarded_agent.invoke(
    Command(resume={"decisions": [{
        "type": "reject",
        "message": "Cancellations require manager sign-off first. Ask the customer to call support.",
    }]}),
    config=config3, version="v2",
)

In [38]:
print(resumed_reject.value["messages"][-1].content)


Booking **BK1042** still requires manager sign-off before it can be canceled. Please contact support for approval.

In [39]:
@tool
def ask_customer(question: str) -> str:
    """Ask the customer a clarifying question and wait for their reply."""
    return "placeholder -- never actually reached, respond intercepts this"


In [43]:

ask_agent = create_agent(
    model=model_advanced,
    tools=[ask_customer],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"ask_customer": {"allowed_decisions": ["respond"]}})],
    checkpointer=InMemorySaver(),
)

In [44]:
config_ask = {"configurable": {"thread_id": "hitl-demo-3"}}


In [45]:
ask_agent.invoke({"messages": [("user", "Cancel booking BK1042")]}, config=config_ask)

{'messages': [HumanMessage(content='Cancel booking BK1042', additional_kwargs={}, response_metadata={}, id='9b144df4-6177-419c-9d86-3241a5ba717c'),
  AIMessage(content='', additional_kwargs={'reasoning_details': [{'data': 'gAAAAABqikDICWO8J3NZ9m8CwOruSZfFQWKhpIHrEUUvJDR4-tTnNqas-W4m2eJBnsAQ-s3JPVU82o-BSPJahT8IsjOURJcCSQkL9RtYtf76p0xcLuk_VU8xSxs8RYIFiuE4Ib3KMMQg51WLO8qC6OdqFmaY7NNIVtkJIIsUN43c_Dni3Z5p6Tm8-xXjASqKAiLpBwbFC_uXXb4D5VMNvGlYkwUCQBdSV3VNhJGWXnhclIHYQkVzIcunNl8YJqYpQ_PslJeWS1CLAOrsrNwTxlgSKNXqJVyk2WkdYCUveacfbN8nSCwCpyGZb3ZB14WSJD67tfsw97jkfPEp6rCR20wtjXbXtdudrUgW-Ejvda2NQj9XEAa-3fjC2XFMNLtRIuGQwPtMWon_78rHzEXUVkmmRMvX5KP2WIdpotXjOWp1LENpLzpUfIGRkIbHwXDbyYJtDPGJPG0T3I3qfTjjMsM3PaA7ddfNGx88hi1WY6D623zUeDytMj9McNa8wTjVJJr_w86hbzGn49HG9W2VwHVXz0AF6uGz9V8bdtRumnaqokR4yYYKe77m-2qLpkWJY5AueDdq98JvjfE9__q03rhQCQ5jUQfb1APX2MBuHoiYkFyl75rwfigvRL8ijh0p0tIahDt-agQt8-jtlyuQvVvrQonxcQjVuEzctvqa6LGOPcY-ZwAw0OtmKz6-IQer0fh0CP029lpgO0H1k-1QxiPBGEHKz_c9mp9Sq8JUglwm_uSMdYo2-hVy54gOIxgFlHEfuSoyL

In [46]:
resumed_agent_respond = ask_agent.invoke(
    Command(resume={"decisions": [{
        "type": "respond",
        "message": "Its Booking BK1042, for the movie interstellar",
    }]}),
    config=config_ask
)

In [47]:
print(resumed_agent_respond)

{
    'messages': [
        HumanMessage(
            content='Cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='9b144df4-6177-419c-9d86-3241a5ba717c'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqikDICWO8J3NZ9m8CwOruSZfFQWKhpIHrEUUvJDR4-tTnNqas-W4m2eJBnsAQ-s3JPVU82o-BSPJahT8IsjOURJcCSQkL9RtYtf76p0xcLu
k_VU8xSxs8RYIFiuE4Ib3KMMQg51WLO8qC6OdqFmaY7NNIVtkJIIsUN43c_Dni3Z5p6Tm8-xXjASqKAiLpBwbFC_uXXb4D5VMNvGlYkwUCQBdSV3VNh
JGWXnhclIHYQkVzIcunNl8YJqYpQ_PslJeWS1CLAOrsrNwTxlgSKNXqJVyk2WkdYCUveacfbN8nSCwCpyGZb3ZB14WSJD67tfsw97jkfPEp6rCR20wt
jXbXtdudrUgW-Ejvda2NQj9XEAa-3fjC2XFMNLtRIuGQwPtMWon_78rHzEXUVkmmRMvX5KP2WIdpotXjOWp1LENpLzpUfIGRkIbHwXDbyYJtDPGJPG0
T3I3qfTjjMsM3PaA7ddfNGx88hi1WY6D623zUeDytMj9McNa8wTjVJJr_w86hbzGn49HG9W2VwHVXz0AF6uGz9V8bdtRumnaqokR4yYYKe77m-2qLpk
WJY5AueDdq98JvjfE9__q03rhQCQ5jUQfb1APX2MBuHoiYkFyl75rwfigvRL8ijh0p0tIahDt-agQt8-jtlyuQvVvrQonxcQjVuEzctvqa6LGOPcY-Z
wAw0OtmKz6-IQer0fh0CP029lpgO0H1k-1QxiPBGEHKz_c9mp9Sq8JUglwm_uSMdYo2-hVy54gOIxgFlHEfuSoyLmDAggunHUNj3o5ZGY225RD4W0YP
cqiWyQCl077fcKBCZJHTEib3KUHRPGaiMmVUvyHBG2f-3HYninxJ1sIaNYxbA-ykzDhY1CF9-I3qC-GUeo6L0wHb3XYWA13AV4r1ISsMnbxF3w2H2Ce
UiAcr5OmS8eXXcbetq3fhyFjwZw5uyHtWnat54EKgXJkennc_zfLI1H3WB9LoWq0KuFCXnWx4pxcuLqMI1vCS1kzEf6iCwwtSa3FGI9gD3UVnDclo6m
BlV1jY2bMI_QS15Os0_7F9k7R87-p4NcSjDunK8nWyLdX-OrZcsUbSj4VjoEzjS8Xkk-HylOvP9UcxUHN4aFvRnDABo18479wmIvMuGJnybE9UN4YFw
lZ5S95Ec2EzL0-ro3GJW_4HbjVBCOcY7hQu-7FlyDjPnBhpDv85pnUYGdhEsRWgiAUHcHHGXhnRPlK6ErDyvyMZSEUgHgTpQoiaexwJX-dp9K2GAyyK
5rdC9UAFH2JrHd1tkI-5aylYjwHEe1xr--QOZDg3d3WT-EnX0p0KZsoHRHlmfN1BtpX2P8Cz6oKj60JbQ_5ycdoMUqGlm41v3mS6ZbxSmDsHdw1V4UZ
-i_591QbF-UGbH6ZDjux-LUND_C_zqrU2cY9KVXjR54z7uiSuv38zmaq7i2i1804xRU1Oq5OfOcGduG51y32z-vgMag_Txz3djTb4PsqwGWrpZD0Ow_
cxfepNQ5fp2NctPa2PSMCx9Mum2EWXLqq5bw7Njp0hZubazhBQftLJ7laPDKjXa9Igm2RLJ2MMxxO6infwXiU6NTAHKZ7z7-C1ufVtm6H8ZoTMRv0in
MiuRmNOSIEhDLbU_3ukb1tmHS2WwmrC-ItmkLsyhtSpMLON0vvPWPIQLKxgvXwYn_I7T1Ot_4aC2CuiinBEeYiim37-4RKhm9LX3-YN0gasHRgdUfsG
RGRW53zozug0gO9OMl5QpyhkGJWc3jLjb427vw_0J3t0_X1Eadq8EtQzaZutKNEz0lCY8h2Nz5pg28AMtGpEeWV0c1fSeR7Q146KstC8FQUxj7dNOEb
oAbUNr5En8HS26yEjf6h-p5EkXI-VPvFm5dyVQLYvUE66lVJl3dXj-ZWcOIjbo7_J6dbsE1gC2hdT6uo8O-VGtXB_4TN1sMTmCvJ01Sm7NWb46Okydt
l5ddXymjy252btoC-DaIAFrTGm9gAnDInMMN0muWefstitjsMpaKybTJrnIH-zyg9eTvlWYtsqzHrZKuvR6FtqXJ_eX81XKRvzylfAgSNgOKG6l2ca1
bvSkYjT_xpt3g==.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_09da9a69f28d5b22016a8a40c83b1487d1b83fb40d6faa615a',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1787445444-CLcQ10VMQvNSHMpPIhYr',
                'created': 1787445444,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0009338,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.000546,
                    'upstream_inference_prompt_cost': 0.0003878,
                    'upstream_inference_cost': 0.0009338
                }
            },
            id='lc_run--01a02c0c-fb88-7fb0-ace1-2269f2241fd0-0',
            tool_calls=[
                {
                    'name': 'ask_customer',
                    'args': {
                        'question': 'To process the cancellation for booking BK1042, please provide the name and 
email address or phone number associated with the booking.'
                    },
                    'id': 'call_c7nPc8J4bvDOTvyKs8gOjXkO',
                    'type': 'tool_call'
  

# Conditional Interrupts

In [48]:
from langchain.agents.middleware import ToolCallRequest

@tool
def cancel_booking_priced(booking_id: str, amount: float) -> str:
    """Cancel a booking with a refund amount."""
    return f"Booking {booking_id} cancelled, ${amount:.2f} refunded."

In [49]:
ToolCallRequest
'''
    tool_call: ToolCall
    tool: BaseTool | None
    state: Any
    runtime: ToolRuntime
'''

'\n    tool_call: ToolCall\n    tool: BaseTool | None\n    state: Any\n    runtime: ToolRuntime\n'

In [50]:
def is_large_refund(request: ToolCallRequest)-> bool:
  # amount = request.tool_call['args'].get('amount',0)
  # return amount >100
  return True

In [54]:
conditional_agent = create_agent(
    model=model_advanced,
    tools=[cancel_booking_priced],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "cancel_booking_priced": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "when": is_large_refund,
                },
            },
        ),
    ],
    checkpointer=InMemorySaver(),
)


In [55]:
config_small = {"configurable": {"thread_id": "small-refund"}}
r_small = conditional_agent.invoke(
    {"messages": [("user", "Cancel booking BK1001, refund $25")]}, config=config_small
)
print("Small refund, paused?", bool(r_small.interrupts))


AttributeError: 'dict' object has no attribute 'interrupts'

In [ ]:
print(r_small)

{
    'messages': [
        HumanMessage(
            content='Cancel booking BK1001, refund $25',
            additional_kwargs={},
            response_metadata={},
            id='e41cd2e6-7d68-4450-ab49-2e86075d1f30'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 96,
                    'prompt_tokens': 142,
                    'total_tokens': 238,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EFYbHYM6rEMPWeni4KvYpwaP5Hm7p',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a027f2-faa7-7e42-ba20-627632f79761-0',
            tool_calls=[
                {
                    'name': 'cancel_booking_priced',
                    'args': {'booking_id': 'BK1001', 'amount': 25},
                    'id': 'call_xJdPpiBwOHaB4mIBohgFWQ8L',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 142,
                'output_tokens': 96,
                'total_tokens': 238,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content='Booking BK1001 cancelled, $25.00 refunded.',
            name='cancel_booking_priced',
            id='2bfad941-99fd-47e2-ad51-369096751ac4',
            tool_call_id='call_xJdPpiBwOHaB4mIBohgFWQ8L'
        ),
        AIMessage(
            content='Done — booking BK1001 has been cancelled and $25.00 refunded.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 19,
                    'prompt_tokens': 193,
                    'total_tokens': 212,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EFYbJbakS60jNbvVoTxeN5aU35tZK',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a027f3-061d-7a22-bd33-a424c552cde9-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 193,
                'output_tokens': 19,
                'total_tokens': 212,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}

In [56]:
config_large = {"configurable": {"thread_id": "large-refund"}}
r_large = conditional_agent.invoke(
    {"messages": [("user", "Cancel booking BK1002, refund $500")]}, config=config_large, version="v2",
)
print("Large refund, paused?", bool(r_large.interrupts))

Large refund, paused? True

In [57]:
print(r_large)

GraphOutput(
    value={
        'messages': [
            HumanMessage(
                content='Cancel booking BK1002, refund $500',
                additional_kwargs={},
                response_metadata={},
                id='845fd36c-de67-46fa-b5fe-aaf64160749a'
            ),
            AIMessage(
                content='',
                additional_kwargs={
                    'reasoning_details': [
                        {
                            'data': 
'gAAAAABqikFR8CDEQsoIv1Hz65LEgxNlK_rgVwOdlAC7d1XDlcE6q7RhRGflPCjdyK1gLHSvYOtwFx0Bkdf17MQbcav25nBXrYkMt1MIKP9YXWOy13
YK4Li2eqplFm9FlhwW4bX9_llIYCpxcyKMzyV_3Mfc8m4TvfTTXeuWoBCMpF7CZVUXKSey2QSW9-g5HuKdQVq9Mdse6gc61cm0CjQHuae4ClUiI17Yi
g_Qcbf7m4LmSYRf5l_pL8HVHJiETUpnYDEXEbQEfqg3pZs06Y4p-8RisxZroUoFC4IAR6_cz240I9Bl3x83sPLVZ_FOj_ICYHgYQVoN7G4MgR9Jg7z_
VXVS58-_Sg5nODpx_iUi3ZHbMVtqsX7vlfnYIQBaerDLGK0HlxZApC-yTThkqJudlmhZGeqSTIDeZy05UEOpgR76sthgIjpa9Ha45KqGAGtb7CkgKTA
WfAKBAsPHokCxacMXiyWGfwnxCEpjkY8xj7OtcRI1a0PBFow3uvNXJG2CCCjO4Zfq24TtpSehN95WeGnWkI107e43OfIuHsEQJD25t4syShmL7gUH32
JWL-dmG0BcbWlCcH0-xWrqDaS9tkjU5DxfU3h1tUTpMfueoYZdB_Xofw2z8EOEpZJLZKXJJNLCi0NhTzhFu8CYBzhC9NoIYz9v-Us2E5UHKOvY_UKSK
fKHJ-H9vJpBjJN6Mbtut-y-qh1-TJVcJVR2qV_9kfkEUWwrFoe-0q5e3sZ49s-zlH1bP5_bRRvLwDDJ-px--w0aon823DqcuVa3mDVAg6kvDeD6H6F_
W2Ak-SiVWsLw8UnXWJNpy5eAdvW5iGl8uAxhlplqD0syCv1Jr6kxc9rtoBiEG5iL4KAIsqMNbUjS-y50C01u5ypAKIRlcIizA2Pox2MhNlaj2Zv1V7g
NQZ4XlK_eaHs6W_brjxr--9NPb-viJl1q7yn8uPUQcoGSpDZMYKTaP9ejQQxd-If5euc3G1Vg7QXNCGwn6EA40Pho8SQ7m9S43gWh0N7eOCncVqA1sa
PCujkrD64XEa8MdlrqotOTvbEyNF4bcKTsjXdXut1W60VYvT-5jwI_QwdKckwHP05i0dBYG9xBIXaRf1sXDEpxh_kTXTQE_i9lDjw6iJXj2wEZk_shd
tCGW9kE_PcT4oaS2Zrr6sCSOqcXAp-v-eWbbqKOMsplRpXitAh0Bglwiw-RQU41g-7BlZcwAYRJuIEpBhkt9iFj49cqV-sQUw6HWWAEFvbHSnxnZPU-
2Wf_bsxL99xLSJfgaI33Ju9huQViSbQp4CVQkUnSfFP1tPu7EFCiQ_q6eDbk8ssJjkSEx4_hsdyEfuMtHlBBfTflBKBAUn6rJHhzzMHkxckWVz9jFuN
3jTJ_8_DPxmqosLACXkLt8qBJC5a9N0jVFzEs_3Jv.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3Blb
mFpIn0',
                            'type': 'reasoning.encrypted',
                            'format': 'openai-responses-v1',
                            'id': 'rs_0012851d1ece51be016a8a4151141887d1944892b10c019759',
                            'index': 0
                        }
                    ]
                },
                response_metadata={
                    'model_name': 'openai/gpt-5.6-luna-pro',
                    'id': 'gen-1787445582-GTQDv6ENEscZyRtowtEh',
                    'created': 1787445582,
                    'object': 'chat.completion',
                    'finish_reason': 'tool_calls',
                    'logprobs': None,
                    'model_provider': 'openrouter',
                    'cost': 0.000562,
                    'cost_details': {
                        'upstream_inference_completions_cost': 0.0002184,
                        'upstream_inference_prompt_cost': 0.0003436,
                        'upstream_inference_cost': 0.000562
                    }
                },
                id='lc_run--01a02c0f-184e-76a1-bac2-6be874f00981-0',
                tool_calls=[
                    {
                        'name': 'cancel_booking_priced',
                        'args': {'booking_id': 'BK1002', 'amount': 500},
                        'id': 'call_BDgBDuv7aJiQPIRDMxflgZWH',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 1718,
                    'output_tokens': 182,
                    'total_tokens': 1900,
                    'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                    'output_token_details': {'reasoning': 80}
                }
            )
        ]
    },
    interrupts=(
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'cancel_booking_priced',
            

In [ ]:
def run_interactive_hitl(agent, config):
    """A real, interactive HITL loop -- uses the CORRECTED edited_action format."""
    result = agent.invoke({"messages": [("user", "Cancel booking BK1042")]}, config=config, version="v2")
    if not result.interrupts:
        print("Nothing paused for review.")
        return

    print("CineBot wants to cancel booking BK1042. Choose a decision:")
    print("  1) approve")
    print("  2) edit     -- cancel a DIFFERENT booking ID instead")
    print("  3) reject")
    choice = input("Type 1, 2, or 3: ").strip()

    if choice == "1":
        decision = {"type": "approve"}
    elif choice == "2":
        new_id = input("Booking ID to cancel instead: ").strip()
        decision = {"type": "edit", "edited_action": {"name": "cancel_booking", "args": {"booking_id": new_id}}}
    elif choice == "3":
        reason = input("Reason: ").strip()
        decision = {"type": "reject", "message": reason}
    else:
        print("Not a valid choice.")
        return

    resumed = agent.invoke(Command(resume={"decisions": [decision]}), config=config, version="v2")
    print()
    print("Final:", resumed.value["messages"][-1].content)

run_interactive_hitl(guarded_agent, {"configurable": {"thread_id": "live-interactive-demo3"}})


CineBot wants to cancel booking BK1042. Choose a decision:

1) approve

2) edit     -- cancel a DIFFERENT booking ID instead

3) reject

Type 1, 2, or 3: 1


Final: Done — booking BK1042 has been cancelled. Would you like a confirmation email sent to the booking email on 
file (or a different address)?